# GCN Over-Smoothing Analysis

**Over-smoothing** is the phenomenon where stacking too many GCN layers causes all node representations to converge to the same vector — making nodes indistinguishable regardless of their true class.

This notebook:
1. Trains GCNs of increasing depth: 1, 2, 3, 5, 8, 10 layers
2. Measures **mean pairwise cosine similarity** of node embeddings
3. Plots similarity vs depth (higher = more over-smoothed)
4. Shows how classification accuracy degrades with excessive depth

**Reference:** Hamilton (2020) Ch. 7; Li et al. (2018) *Deeper Insights into GCNs*.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import torch
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120

from vanilla_gcn.config import load_config
from vanilla_gcn.seed import set_seed
from vanilla_gcn.data.synthetic import create_synthetic_graph
from vanilla_gcn.data.preprocessing import prepare_graph
from vanilla_gcn.models.vanilla_gcn import VanillaGCN
from vanilla_gcn.training.trainer import train_gcn
from vanilla_gcn.analysis.oversmoothing import (
    run_oversmoothing_experiment,
    plot_oversmoothing,
    mean_pairwise_cosine_similarity,
)
from vanilla_gcn.visualization.embeddings import visualize_embeddings
from vanilla_gcn.visualization.layers import visualize_layer_embeddings
from vanilla_gcn.utils.logging import setup_logging

setup_logging('WARNING', force=True)  # quiet for sweep
cfg = load_config(ROOT / 'configs' / 'default.yaml')
print('Ready. Seed:', cfg.seed)

## 1. Data

In [ ]:
set_seed(cfg.seed)
data = create_synthetic_graph(seed=cfg.seed)
print(data)

## 2. Why Over-Smoothing Happens

Each GCN layer applies:
$$H^{(k)} = \sigma(\tilde{A}\, H^{(k-1)}\, W^{(k)})$$

The matrix $\tilde{A}$ is a **low-pass graph filter** — it smooths node features by mixing each node's representation with its neighbors. After $K$ applications:

- $K=1$: node uses its own + direct neighbor features
- $K=2$: node uses 2-hop neighborhood  
- $K \to \infty$: all nodes converge to the **same** stationary distribution

This loss of diversity = **over-smoothing**.

## 3. Similarity Metric

We measure over-smoothing as **mean pairwise cosine similarity** between all node embeddings:

$$\text{similarity} = \frac{1}{N(N-1)} \sum_{i \neq j} \frac{z_i \cdot z_j}{\|z_i\|\,\|z_j\|}$$

- Value near **0** → diverse embeddings (good for classification)
- Value near **1** → all embeddings identical (over-smoothed)

## 4. Depth Sweep Experiment

In [ ]:
DEPTHS = [1, 2, 3, 5, 8, 10]

result = run_oversmoothing_experiment(
    A=data.adjacency,
    X=data.features,
    y=data.labels,
    train_mask=data.train_mask,
    val_mask=data.val_mask,
    depths=DEPTHS,
    hidden_dim=cfg.model.hidden_dim,
    epochs=cfg.training.epochs,
    lr=cfg.training.learning_rate,
    weight_decay=cfg.training.weight_decay,
    seed=cfg.seed,
)
print(result.summary())

## 5. Over-Smoothing Plot

In [ ]:
fig = plot_oversmoothing(
    result,
    save_path=ROOT / 'outputs/figures/03_oversmoothing.png',
)
plt.show()

## 6. Visual Comparison: 2-Layer vs 10-Layer Embeddings

In [ ]:
def get_embeddings_for_depth(depth, seed=42):
    """Train a GCN of given depth and return intermediate embeddings."""
    set_seed(seed)
    _, A_tilde, X, y = prepare_graph(data.adjacency, data.features, data.labels)
    m = VanillaGCN(
        input_dim=data.num_features,
        hidden_dim=cfg.model.hidden_dim,
        num_classes=data.num_classes,
        num_layers=depth,
        debug=False,
    )
    train_gcn(m, A_tilde, X, y, data.train_mask, data.val_mask,
              epochs=cfg.training.epochs, lr=0.01, log_every=200)
    return m.get_embeddings(X, A_tilde)

embs_2  = get_embeddings_for_depth(2)
embs_10 = get_embeddings_for_depth(10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Node Embeddings: 2 Layers vs 10 Layers (PCA)', fontsize=14, fontweight='bold')

# Select the last hidden layer for both
def last_hidden(embs):
    hidden_keys = [k for k in embs if k not in ('input', 'logits')]
    return embs[hidden_keys[-1]] if hidden_keys else embs['input']

from vanilla_gcn.visualization.embeddings import visualize_embeddings
visualize_embeddings(
    last_hidden(embs_2), data.labels, data.node_ids,
    title='2-Layer GCN (diverse)', ax=ax1,
)
visualize_embeddings(
    last_hidden(embs_10), data.labels, data.node_ids,
    title='10-Layer GCN (over-smoothed)', ax=ax2,
)
plt.tight_layout()
fig.savefig(ROOT / 'outputs/figures/03_embedding_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## 7. Interpretation

**What the plots show:**
- **2-layer GCN**: Node embeddings from different classes are well separated in PCA space → low similarity → good classification
- **10-layer GCN**: Embeddings collapse toward a single cluster → high similarity → classification degrades

**Key insight:** On this small graph, over-smoothing appears quickly because the graph diameter is small — only a few hops are needed to reach any node from any other.

**Practical guidance:**
- For most real-world graphs: **2–3 layers** is optimal for vanilla GCN
- Deeper architectures require techniques like **DropEdge**, **PairNorm**, **GCNII**, or **residual connections** to mitigate over-smoothing
- The optimal depth depends on the graph structure and task — always validate!

> **Note:** This notebook demonstrates an *illustrative* over-smoothing experiment on a tiny synthetic graph. The exact depth at which performance degrades will differ on real-world datasets.

## 8. Similarity Values Summary

In [ ]:
print(f'{'Depth':>6} | {'Similarity':>12} | {'Train Acc':>10} | {'Val Acc':>8}')
print('-' * 45)
for d, s, ta, va in zip(result.depths, result.similarities, result.train_accs, result.val_accs):
    bar = '█' * int(s * 20)
    print(f'{d:>6} | {s:>12.4f} | {ta:>9.1f}% | {va:>7.1f}%  {bar}')